<a href="https://colab.research.google.com/github/HojuneLee0106/Performace_Improvement_Contest/blob/main/result_generator_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""카카오 약관 4종 (출제 기준일 2026-08-04 적용본) 조 단위 원문.

노트북 셀에 그대로 붙여넣는다. 외부 네트워크·Drive·사전 업로드 없이 복원된다.
시행일: 계정약관/통합서비스약관 2026-05-29, 위치정보 2026-07-16, 통합약관 2022-08-25.

이 데이터는 build_terms.py 가 PDF 원문에서 생성했다. 조 헤더는 "그 줄 전체가 헤더인
경우"만 인정한다. 이전 파서는 제5조 본문 첫 문장 "제 4 조에 따른 가입 신청자에게 ..."
를 헤더로 오인해 통합서비스약관 제4조(계약의 성립)를 통째로 날렸고, 그 자리에 제5조
본문이 복제돼 들어갔다. 아래 _assert_corpus() 가 같은 사고를 실행 시점에 잡는다.
"""
import base64, gzip, json, re

_TERMS_B64 = (
"..."
"k5MJ3QEA"
)

TERMS = json.loads(gzip.decompress(base64.b64decode(_TERMS_B64)).decode("utf-8"))


# ─────────────────────────────────────────────────────────────────────
# 코퍼스 무결성 검증 — 색인·검색보다 먼저 돈다.
# 여기서 죽는 편이, 근거 없는 답을 그럴듯하게 만들어 제출하는 것보다 낫다.
# ─────────────────────────────────────────────────────────────────────
_EXPECTED_ARTICLES = {
    "카카오계정 약관": 17,
    "카카오 위치정보 이용약관": 16,
    "카카오 통합서비스약관": 18,
    "카카오 통합 약관": 21,
}

# 조 번호가 밀리거나 본문이 뒤바뀌면 즉시 깨지는 지점들.
_CANARY = {
    ("카카오 통합서비스약관", 4): "카카오계정을 먼저 생성",
    ("카카오 통합서비스약관", 5): "가입의 제한",
    ("카카오 통합서비스약관", 7): "2 시간 이상 복구가 지연",
    ("카카오계정 약관", 7): "SSO(Single Sign On)",
    ("카카오계정 약관", 10): "담당자 1인만 이용",
    ("카카오 위치정보 이용약관", 6): "그 사유 및 제한기간",
    ("카카오 위치정보 이용약관", 8): "6개월간 보관",
    ("카카오 위치정보 이용약관", 12): "보호의무자임을 증명하는 서면",
    ("카카오 통합 약관", 3): "세부지침에 따릅니다",
    ("카카오 통합 약관", 10): "탈퇴한 후에도 존속",
}

_HEAD_NO = re.compile(r"^\s*제\s*(\d{1,3})\s*조")
# build_terms.py 와 동일한 '줄 전체가 헤더' 규칙
_FULL_HEAD = re.compile(
    r"^\s*제\s*\d{1,3}\s*조"
    r"(?:\s*[(（][^)）\n]{1,60}[)）]|\s+[^\s(（][^\n]{0,40}|\s*)"
    r"[\s·・.,]*$"
)


def _assert_corpus(terms):
    errs = []
    for doc, expected in _EXPECTED_ARTICLES.items():
        if doc not in terms:
            errs.append("%s: 문서가 없습니다" % doc)
            continue
        nums = sorted(int(k) for k in terms[doc])

        # (1) 조 번호 1..N 연속 + 개수
        if nums != list(range(1, expected + 1)):
            errs.append("%s: 조 번호가 %s (기대 1..%d)" % (doc, nums, expected))

        seen = {}  # 조번호 -> 공백 제거한 본문
        for n in nums:
            body = terms[doc][str(n)]
            head, _, rest = body.partition("\n")

            # (2) 첫 줄이 '줄 전체가 헤더' 형태이고 그 번호가 키와 같은가
            #     "제 4 조에 따른 가입 신청자에게 ..." 처럼 본문이 이어지는 줄은
            #     _HEAD_NO 만으로는 통과해 버린다. _FULL_HEAD 로 막는다.
            m = _HEAD_NO.match(head)
            if not _FULL_HEAD.match(head):
                errs.append("%s 제%d조: 첫 줄이 조 헤더가 아니라 본문입니다 → %r"
                            % (doc, n, head[:50]))
            elif int(m.group(1)) != n:
                errs.append("%s 제%d조: 첫 줄이 제%s조 → %r" % (doc, n, m.group(1), head[:40]))

            # (3) 본문 유실
            if len(rest.strip()) < 20:
                errs.append("%s 제%d조: 본문이 비었습니다(%d자)" % (doc, n, len(rest.strip())))

            # (4) 같은 문서 안에서 본문이 겹치면 잘못 분할된 것.
            #     완전 일치만 보면 부족하다. 잘못 쪼개진 조각은 원본 조의
            #     '부분집합'으로 들어오기 때문에 포함 관계까지 본다.
            key = re.sub(r"\s+", "", rest)
            if len(key) >= 40:
                for other, okey in seen.items():
                    if key in okey or okey in key:
                        errs.append("%s 제%d조: 제%d조와 본문이 겹칩니다" % (doc, n, other))
                        break
                seen[n] = key

            # (5) 본문에 다른 조 헤더가 통째로 섞임 = 병합
            for line in rest.split("\n"):
                if _FULL_HEAD.match(line):
                    errs.append("%s 제%d조: 본문에 조 헤더가 섞임 → %r" % (doc, n, line[:40]))

    # (6) 카나리 — 알려진 문장이 지정한 조에 있는가
    for (doc, n), needle in _CANARY.items():
        if needle not in terms.get(doc, {}).get(str(n), ""):
            errs.append("%s 제%d조: 카나리 %r 없음" % (doc, n, needle))

    if errs:
        raise AssertionError(
            "약관 코퍼스 검증 실패 %d건 — 색인을 만들지 않고 중단합니다.\n  · "
            % len(errs) + "\n  · ".join(errs)
        )


_assert_corpus(TERMS)

ARTICLES = [(doc, int(no), text) for doc, arts in TERMS.items() for no, text in arts.items()]
print("[코퍼스] %d개 조항 · 검증 6종 통과" % len(ARTICLES))
# ── 셀 0.5 — 통합 약관 -layout 따옴표 잔재 정리 ─────────────────────────────
# -layout 추출이 따옴표를 자리 밖으로 밀어냈다: "등(이하 세부지침 ) ‘ ’ 의"
# P08 이 이 깨진 '세부지침' 정의 문장 때문에 결론을 뒤집었다.
# 떠도는 따옴표를 지우고 (이하 X) 정의부를 다시 감싼다. 의미 변경 없음.
# 반드시 셀 1(색인)보다 먼저 실행한다.
import re

_ORPHAN = re.compile(r"\s*[‘’“”]\s*[‘’“”]\s*")   # ' ' 처럼 붙어 떠도는 쌍
_LONE = re.compile(r"\s[‘’“”]\s")                  # 홀로 떠도는 따옴표
_IHA = re.compile(r"\((이하\s*(?:총칭하여\s*)?)([^()‘’]*?)\s*\)")


def _normalize_quotes(t):
    t = _ORPHAN.sub(" ", t)
    t = _LONE.sub(" ", t)
    t = _IHA.sub(lambda m: "(" + m.group(1) + "‘" + m.group(2).strip() + "’)", t)
    t = re.sub(r"\(\s+", "(", t)
    t = re.sub(r"\s+\)", ")", t)
    t = re.sub(r"\)\s+(의|를|을|는|은|에)\b", r")\1", t)
    return re.sub(r"[ \t]{2,}", " ", t)


for _no in ["1", "3", "9", "10"]:
    TERMS["카카오 통합 약관"][_no] = _normalize_quotes(TERMS["카카오 통합 약관"][_no])

ARTICLES = [(doc, int(no), text) for doc, arts in TERMS.items() for no, text in arts.items()]

# 이 두 줄이 통과해야 이번 수정이 실제로 반영된 것이다
assert "(이하 ‘세부지침’)의 규정에 따릅니다" in TERMS["카카오 통합 약관"]["3"]
assert "전 세계적이고 영구적인" in TERMS["카카오 통합 약관"]["10"]
print("[패치] 통합 약관 따옴표 잔재 정리 완료 — 제1·3·9·10조")

# -*- coding: utf-8 -*-
"""조 단위 BM25 색인 — 토크나이저 3종 비교용.

의존성 0. 새 Colab 세션에서 pip 없이 그대로 돈다.
1번 셀 최상단(전역)에서 한 번만 실행하고, answer_question 은 search() 만 호출한다.
"""
import math
import re
from collections import Counter


# ─────────────────────────────────────────────────────────────────────
# 토크나이저 — 실험 변수
# ─────────────────────────────────────────────────────────────────────
_WORD = re.compile(r"[가-힣]+|[a-zA-Z]+|\d+")

def tok_space(text):
    """baseline. 조사가 붙은 채로 토큰이 되므로 '위치정보를' != '위치정보는'."""
    return _WORD.findall(text.lower())


def tok_bigram(text):
    """한글은 문자 bigram, 영숫자는 단어 단위.

    법령체는 핵심어가 대부분 한자어 명사라 bigram 조각이 변별력을 유지하고,
    조사 차이는 마지막 bigram 하나로 흡수된다.
    """
    out = []
    for w in _WORD.findall(text.lower()):
        if "가" <= w[0] <= "힣":
            if len(w) == 1:
                out.append(w)
            else:
                out.extend(w[i:i + 2] for i in range(len(w) - 1))
        else:
            out.append(w)
    return out


def make_tok_kiwi():
    """명사·어간만 추출. 설치 실패 시 None 을 돌려주고 실험에서 빠진다."""
    try:
        from kiwipiepy import Kiwi
    except ImportError:
        return None
    kiwi = Kiwi()
    keep = ("NNG", "NNP", "NNB", "NR", "SL", "SN", "VV", "VA", "XR")
    def tok(text):
        return [t.form for t in kiwi.tokenize(text) if t.tag in keep]
    return tok


# ─────────────────────────────────────────────────────────────────────
# BM25
# ─────────────────────────────────────────────────────────────────────
class BM25:
    def __init__(self, docs, tokenize, k1=1.2, b=0.75):
        """docs: [(문서명, 조번호, 본문)]"""
        self.meta = [(d, n) for d, n, _ in docs]
        self.tokenize = tokenize
        self.k1, self.b = k1, b

        self.tf = [Counter(tokenize(t)) for _, _, t in docs]
        self.len = [sum(c.values()) for c in self.tf]
        self.avglen = sum(self.len) / len(self.len)

        df = Counter()
        for c in self.tf:
            df.update(c.keys())
        N = len(docs)
        # +1 로 하한을 둔다. N=72 라 흔한 토큰이 음수 IDF 가 되면
        # 그 토큰을 가진 조항이 오히려 감점되는 역전이 생긴다.
        self.idf = {t: math.log(1 + (N - n + 0.5) / (n + 0.5)) for t, n in df.items()}

    def scores(self, query):
        q = self.tokenize(query)
        out = [0.0] * len(self.tf)
        for term in q:
            idf = self.idf.get(term)
            if idf is None:
                continue
            for i, c in enumerate(self.tf):
                f = c.get(term)
                if not f:
                    continue
                denom = f + self.k1 * (1 - self.b + self.b * self.len[i] / self.avglen)
                out[i] += idf * f * (self.k1 + 1) / denom
        return out

    def search(self, query, top_k=4, allowed_docs=None):
        """allowed_docs 가 주어지면 그 문서 안에서만 고른다."""
        sc = self.scores(query)
        idx = range(len(sc))
        if allowed_docs:
            idx = [i for i in idx if self.meta[i][0] in allowed_docs]
        ranked = sorted(idx, key=lambda i: -sc[i])[:top_k]
        return [(self.meta[i][0], self.meta[i][1], sc[i]) for i in ranked]


# ─────────────────────────────────────────────────────────────────────
# 색인 대상 텍스트 — 조 제목 가중
# ─────────────────────────────────────────────────────────────────────
_TITLE = re.compile(r"^제\s*\d+\s*조\s*[(（]?([^)）\n]*)")

def index_text(doc_name, art_no, body, title_boost=3):
    """제목은 본문보다 짧지만 변별력이 훨씬 높다. 반복해 tf 를 올린다.

    문서명도 함께 넣어야 '카카오계정 약관에서…' 처럼 약관명이 박힌 질문에
    해당 문서 조항이 가산점을 받는다.
    """
    m = _TITLE.match(body.strip())
    title = m.group(1).strip() if m else ""
    return " ".join([doc_name] + [title] * title_boost + [body])

# -*- coding: utf-8 -*-
# =====================================================================================
#  셀 3 — BM25 색인 (bm25_index.py 내용을 붙인 뒤, 아래를 이어서 실행)
# =====================================================================================
#  색인은 세션당 한 번만 만든다. answer_question 안에서 만들면 문항마다 재구축된다.

CORPUS = [(d, n, index_text(d, n, t)) for d, n, t in ARTICLES]
BM = BM25(CORPUS, tok_bigram)

# 프롬프트에 넣을 원문. 색인용 텍스트(index_text)는 제목이 3번 반복돼 있어
# 검색에는 좋지만 모델에게 보여주기엔 지저분하다. 원문은 따로 들고 간다.
ART_TEXT = {(d, n): t for d, n, t in ARTICLES}

print(f"[색인] {len(CORPUS)}개 조항 · 어휘 {len(BM.idf):,}개 · 평균 길이 {BM.avglen:.0f} 토큰")

# 색인 점검 — 여기서 1위가 엉뚱하면 뒤 단계가 전부 무의미하다
for q in ["사업자/단체 카카오계정은 담당자 몇 명이 이용할 수 있나요?",
          "8세 이하 아동의 보호의무자가 동의하려면 어떤 서류를 제출해야 하나요?"]:
    d, n, sc = BM.search(q, top_k=1)[0]
    print(f"  · {d} 제{n}조 ({sc:.1f})")


import importlib, importlib.util, subprocess, sys, time
import torch

# ── bitsandbytes 확보 ────────────────────────────────────────────────────────
if importlib.util.find_spec("bitsandbytes") is None:
    # --no-deps 필수: 이게 없으면 pip가 torch/numpy를 갈아끼워 재시작이 강제된다
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    "bitsandbytes"], check=True)
    importlib.invalidate_caches()          # 재시작 대신
import bitsandbytes
print(f"[env] bitsandbytes {bitsandbytes.__version__} · torch {torch.__version__}")

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

TOKENIZER = AutoTokenizer.from_pretrained(MODEL_NAME)

BNB = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,   # T4는 bf16 미지원. 반드시 fp16
)

_t = time.time()
MODEL = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BNB,
    torch_dtype=torch.float16,
    device_map="cuda:0",          # "auto"는 CPU offload가 섞일 수 있어 고정
    attn_implementation="sdpa",   # flash_attention_2는 Ampere 이상 전용
)
MODEL.eval()
print(f"[로드] {time.time()-_t:.0f}s · {torch.cuda.memory_allocated()/1e9:.1f}GB")

def _build_cjk_ban(tokenizer):
    ban = []
    for tid in range(len(tokenizer)):
        s = tokenizer.decode([tid])
        if any("\u4e00" <= ch <= "\u9fff" for ch in s):
            ban.append([tid])
    return ban


CJK_BAN = _build_cjk_ban(TOKENIZER)
print(f"[언어] 한자 토큰 {len(CJK_BAN):,}개 차단")

def generate(system: str, user: str, max_new_tokens: int = 512) -> str:
    text = TOKENIZER.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = TOKENIZER([text], return_tensors="pt").to(MODEL.device)
    with torch.inference_mode():
        out = MODEL.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            bad_words_ids=CJK_BAN,     # 한자 토큰을 후보에서 제외
            pad_token_id=TOKENIZER.eos_token_id,
        )
    return TOKENIZER.decode(out[0][inputs.input_ids.shape[-1]:],
                            skip_special_tokens=True).strip()


# 워밍업 — 첫 CUDA 커널 컴파일을 여기서 끝낸다
_t = time.time()
_ = generate("너는 한국어로 답한다.", "안녕하세요. 짧게 인사해 주세요.", max_new_tokens=32)
print(f"[워밍업] {time.time()-_t:.1f}s · {torch.cuda.memory_allocated()/1e9:.1f}GB")

# -*- coding: utf-8 -*-
# =====================================================================================
#  셀 5 — answer_question + 고정 FastAPI 연결
# =====================================================================================
#  앞선 셀에서 만들어진 전역만 조회한다. 여기서 색인을 만들거나 모델을 로드하지 않는다.
#    셀 2: TERMS, ARTICLES
#    셀 3: BM(BM25 색인), ART_TEXT
#    셀 4: generate()
# =====================================================================================

import re
import subprocess
import sys
import threading

# ── 검색 설정 ────────────────────────────────────────────────────────────────────────
SCORE_RATIO = 0.75   # 1위 점수 대비 이 비율 미만이면 버린다.
                     # 공개 10문항 기준 문항당 평균 2.3개가 남고, 중복 조항이 있는
                     # 문항(P02: 서비스 중단 2시간 공지)에서만 3개가 살아남는다.
MAX_EVIDENCE = 4     # 러너 계약 상한. 초과하면 그 문항이 통째로 폐기된다.
MAX_NEW_TOKENS = 512

ALLOWED_DOCS = {
    "카카오계정 약관",
    "카카오 위치정보 이용약관",
    "카카오 통합서비스약관",
    "카카오 통합 약관",
}
SYSTEM = """너는 카카오 약관 조항을 근거로 질문에 한국어로 답하는 도우미다.

규칙:
1. 반드시 아래 제시된 약관 조항 원문만 근거로 삼는다. 원문에 없는 내용은 절대 지어내지 않는다.
2. 질문이 여러 가지를 묻거나 원문이 항목을 열거하면 빠짐없이 모두 답한다.
   질문이 A, B, C를 물으면 A, B, C 각각에 대해 답한다.
   절차나 순서를 묻는 질문은 화살표나 기호로 압축하지 말고,
   각 단계를 누가 무엇을 하는지 원문의 표현대로 문장으로 풀어 쓴다.
3. 질문이 "예" 또는 "아니오"로 답할 수 있는 가부 질문일 때만 첫 문장에서
   예/아니오를 밝힌다. 그 외에는 묻는 내용을 바로 답한다.
   질문 문장을 그대로 옮겨 적지 않는다.
   예/아니오는 반드시 근거 원문의 내용과 일치해야 한다.
4. 숫자, 기간, 조문 번호, 서류 이름은 원문에 적힌 그대로 옮긴다.
   답변 문장도 근거 원문의 표현과 어순을 최대한 그대로 사용한다.
   같은 뜻의 다른 말로 바꿔 쓰지 않는다.
5. 근거 조항에 답이 없으면 "제시된 약관 조항에서 확인할 수 없습니다"라고만 답한다.
6. 인사말이나 서두 없이 답변 본문만 쓴다.
   답을 담고 있는 조항에 그 답과 직접 이어지는 문장이 더 있으면,
   원문의 표현을 그대로 살려 한 문장 덧붙인다.
   근거 조항에 없는 내용은 어떤 경우에도 덧붙이지 않는다.
7. 질문이 특정 결론을 전제하고 있더라도, 원문이 그와 다르면 원문을 따른다.
   가부 질문에 답하기 전에 원문에서 해당 문장을 먼저 찾아 확인한다.
   """


USER_TEMPLATE = """[약관 조항]
{context}

[질문]
{question}

위 조항을 근거로 질문에 답하시오."""


_DOC_PATTERNS = [
    ("카카오 통합서비스약관",   re.compile(r"통합\s*서비스\s*약관")),
    ("카카오 위치정보 이용약관", re.compile(r"위치정보\s*이용\s*약관")),
    ("카카오계정 약관",         re.compile(r"카카오\s*계정\s*약관")),
    ("카카오 통합 약관",        re.compile(r"카카오\s*통합\s*약관")),
]


def named_docs(question):
    """'카카오 통합 약관'은 '카카오 통합서비스약관'에 매칭되지 않는다(사이에 '서비스')."""
    return {d for d, p in _DOC_PATTERNS if p.search(question)}


def select_evidence(question):
    allowed = named_docs(question) or None
    ranked = BM.search(question, top_k=MAX_EVIDENCE, allowed_docs=allowed)
    if not ranked:
        return []
    top = ranked[0][2]
    if top <= 0:
        return [(ranked[0][0], ranked[0][1])]
    return [(d, n) for d, n, sc in ranked if sc >= SCORE_RATIO * top]


_CIRCLED = "①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮"
_PARA_CIRCLED = re.compile(r"^\s*([%s])\s*" % _CIRCLED)
_PARA_DIGIT = re.compile(r"^\s*(\d{1,2})\.\s+")


def split_paragraphs(doc, body):
    """조 본문을 [(항 라벨 or None, 텍스트)] 로 나눈다.

    항 표기가 없는 조(예: 통합 약관 제3조)는 통째로 한 덩어리로 돌려준다.
    라벨을 억지로 붙이면 없는 구조를 지어내는 셈이 된다.
    """
    lines = body.strip().split("\n")
    head, rest = lines[0], lines[1:]

    pat = _PARA_DIGIT if doc == "카카오 통합 약관" else _PARA_CIRCLED

    blocks, label, buf = [], None, []
    for line in rest:
        m = pat.match(line)
        if m:
            if buf:
                blocks.append((label, "\n".join(buf).strip()))
            label, buf = m.group(1), [line]
        else:
            buf.append(line)
    if buf:
        blocks.append((label, "\n".join(buf).strip()))

    return head, [b for b in blocks if b[1]]


def build_context(evidence):
    blocks = []
    for i, (doc, no) in enumerate(evidence, 1):
        body = ART_TEXT[(doc, no)]
        body = re.sub(r"[ \t]{2,}", " ", body)      # pdftotext 정렬 공백 정리
        head, paras = split_paragraphs(doc, body)

        parts = [f"({i}) {doc}", head.strip()]
        for label, text in paras:
            if label is None:
                parts.append(text)
            else:
                parts.append(f"[제{no}조 제{label}항]\n{text}")
        blocks.append("\n".join(parts))
    return "\n\n".join(blocks)

def answer_question(question: str):
    """공통 러너가 질문마다 호출하는 고정 진입점.

    반환 계약: {"answer": str, "retrieved": [[문서명, 조번호], ...]}  근거 1~4개.
    """
    if not isinstance(question, str) or not question.strip():
        raise ValueError("question은 비어 있지 않은 문자열이어야 합니다.")
    question = question.strip()

    evidence = select_evidence(question)
    answer = ""
    if evidence:
        try:
            answer = generate(
                SYSTEM,
                USER_TEMPLATE.format(context=build_context(evidence), question=question),
                max_new_tokens=MAX_NEW_TOKENS,
            )
        except Exception as exc:      # 생성이 죽어도 검색 점수는 건진다
            print(f"[생성 실패] {type(exc).__name__}: {exc}", flush=True)

    # ── 반환 직전 방어 ───────────────────────────────────────────────────────────────
    # retrieved 가 0개거나 5개 이상이면 러너가 ValueError 를 내고 그 문항을
    # answer:"" 로 통째로 버린다. 조번호가 numpy 정수면 FastAPI 직렬화에서 500 이
    # 나는데, 그건 러너의 방어망(_sp_json_safe_art)보다 앞단이라 걸러주지 못한다.
    retrieved = [[str(d), int(n)] for d, n in evidence if d in ALLOWED_DOCS][:MAX_EVIDENCE]
    if not retrieved:
        retrieved = [["카카오 통합 약관", 1]]
    if not isinstance(answer, str) or not answer.strip():
        answer = "제시된 약관 조항에서 확인할 수 없습니다."

    return {"answer": answer.strip(), "retrieved": retrieved}


# =====================================================================================
#  고정 FastAPI 연결 영역 — 기본 틀 그대로. 경로와 응답 형식을 바꾸지 않는다.
# =====================================================================================
def _install_server_packages():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn"],
        check=True,
    )


_install_server_packages()

from fastapi import FastAPI, HTTPException  # noqa: E402

app = FastAPI(title="KTB AI Performance Result Generator")
_GENERATION_LOCK = threading.Lock()


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/answer")
def answer_api(payload: dict):
    question = payload.get("question")
    if not isinstance(question, str) or not question.strip():
        raise HTTPException(status_code=400, detail="question must be a non-empty string")
    with _GENERATION_LOCK:
        return answer_question(question.strip())


print("[셀 5 준비] answer_question + FastAPI 등록 완료. 이제 2번 공통 러너를 실행하세요.")

[코퍼스] 72개 조항 · 검증 6종 통과
[패치] 통합 약관 따옴표 잔재 정리 완료 — 제1·3·9·10조
[색인] 72개 조항 · 어휘 2,592개 · 평균 길이 378 토큰
  · 카카오계정 약관 제10조 (38.7)
  · 카카오 위치정보 이용약관 제12조 (40.0)
[env] bitsandbytes 0.50.0 · torch 2.11.0+cu128


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[로드] 471s · 5.6GB
[언어] 한자 토큰 25,557개 차단
[워밍업] 1.9s · 5.6GB
[셀 5 준비] answer_question + FastAPI 등록 완료. 이제 2번 공통 러너를 실행하세요.


In [ ]:
# 2번 셀 — 공개 10문항 답변 파일 생성
# 이 셀은 전 팀 공통이며 _SP_TEAM 한 줄 외에는 수정하지 않습니다.
# 새 Google Colab T4 런타임에서 결과기 코드를 먼저 실행한 뒤 이 셀을 실행합니다.
#
# 사용 순서
# 1. 새 Google Colab T4 런타임에서 1번 셀 결과기 코드를 실행합니다.
# 2. 이 공통 러너를 2번 셀에 그대로 둡니다.
# 3. 맨 위 _SP_TEAM에 운영진이 알려준 숫자 팀 식별자를 입력합니다.
# 4. 생성된 answers_public_<팀>.json을 결과기 코랩 파일과 함께 제출합니다.
# 공개 문항 10개 · 실행 방식: http
# ═══════════════════════════════════════════════════════════════
#  ★ 여기 한 줄만 자기 팀으로 바꾸세요. 나머지는 손대지 마세요. ★
# ═══════════════════════════════════════════════════════════════
_SP_TEAM = "13"          # 예: "1"  ← 운영진이 알려준 팀 식별자(숫자)를 그대로 적습니다
# ═══════════════════════════════════════════════════════════════

import builtins as _sp_builtins
import json as _sp_json
import os as _sp_os_rt
import re as _sp_re
import signal as _sp_signal
import socket as _sp_socket
import sys as _sp_sys
import time as _sp_time
import traceback as _sp_traceback
import unicodedata as _sp_unicodedata
import urllib.error as _sp_urlerror
import urllib.request as _sp_urlrequest

_sp_open = _sp_builtins.open
_sp_print = _sp_builtins.print

if "_sp_real_sys_exit" in globals():
    _sp_sys.exit = _sp_real_sys_exit
    if _sp_real_exit is not None:
        _sp_builtins.exit = _sp_real_exit
    if _sp_real_quit is not None:
        _sp_builtins.quit = _sp_real_quit

_SP_OUTPUT_DIR = "/content/"
_SP_OUTPUT_PREFIX = "answers_public_"
_SP_EXPECTED_OUTPUT_PATH = ""
_SP_TEAM_ALLOWED = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz_-"
_SP_TEAM_MAX_LEN = 32
_SP_TEAM_NUMERIC_ONLY = True

def _sp_team_howto(head):
    """중단 사유 + 학생이 바로 고칠 수 있는 안내를 한 덩어리로 만든다."""
    rule = (
        "1 이상의 정수를 문자열로 입력합니다. 예: 1, 2, 17"
        if _SP_TEAM_NUMERIC_ONLY
        else "영문·숫자·밑줄(_)·하이픈(-) 1~" + str(_SP_TEAM_MAX_LEN) + "자"
    )
    return (
        head
        + "\n"
        + "\n  [고치는 법] 이 셀 맨 위 ★ 상자 안의 한 줄을 이렇게 바꾸세요."
        + '\n      _SP_TEAM = "1"      ← 운영진이 알려준 팀 식별자(숫자)를 따옴표 안에 그대로'
        + "\n  [쓸 수 있는 값] " + rule
        + "\n                 띄어쓰기와 / \\ . : 같은 경로 문자는 파일 이름을 깨뜨려 쓸 수 없습니다."
        + "\n  [왜] 결과 파일 이름이 " + _SP_OUTPUT_PREFIX + "<팀>.json 이고, 채점은 이 이름으로"
        + "\n       어느 팀 답안인지 가립니다. 비워 두면 채점 자체가 되지 않습니다."
    )

def _sp_resolve_team(value):
    """_SP_TEAM 을 검사·정리해 돌려준다. 쓸 수 없는 값이면 RuntimeError 로 즉시 중단."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)가 비어 있어 실행을 중단했습니다. 결과 파일은 만들지 않았습니다."))
    team = value.strip()
    if _SP_TEAM_NUMERIC_ONLY and not _sp_re.fullmatch(r"[1-9][0-9]*", team):
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)는 운영진이 알려준 숫자여야 합니다. 지금 값: " + repr(value)))
    if len(team) > _SP_TEAM_MAX_LEN:
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)가 너무 깁니다(" + str(len(team)) + "자). 팀 이름이 아니라 짧은 식별자입니다."))
    _bad = _sp_builtins.sorted(
        _sp_builtins.set(c for c in team if c not in _SP_TEAM_ALLOWED and not ("가" <= c <= "힣")))
    if _bad:
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)에 파일 이름으로 쓸 수 없는 문자가 있습니다: "
            + ", ".join(repr(c) for c in _bad) + "   (지금 값: " + repr(value) + ")"))
    return team

_SP_TEAM = _sp_resolve_team(_SP_TEAM)
if any(ord(c) > 127 for c in _SP_TEAM):
    _sp_print("[주의] 팀 식별자에 한글 등 ASCII 밖 문자가 있습니다: " + _SP_TEAM
              + " — 운영진이 알려준 식별자가 맞는지 확인하세요."
              " 한글 파일 이름은 내려받기·올리기 과정에서 자모 표현이 달라져 팀이 어긋날 수 있습니다.",
              flush=True)

_SP_OUTPUT_PATH = _SP_OUTPUT_DIR.rstrip("/") + "/" + _SP_OUTPUT_PREFIX + _SP_TEAM + ".json"
if _SP_EXPECTED_OUTPUT_PATH and (_sp_os_rt.path.basename(_SP_OUTPUT_PATH)
                                 != _sp_os_rt.path.basename(_SP_EXPECTED_OUTPUT_PATH)):
    raise RuntimeError(
        "이 셀은 " + _sp_os_rt.path.basename(_SP_EXPECTED_OUTPUT_PATH) + " 용으로 생성됐는데 "
        + _sp_os_rt.path.basename(_SP_OUTPUT_PATH) + " 로 저장하려 합니다"
        "(_SP_TEAM 을 손으로 고쳤습니까?). 다른 팀으로 돌리려면 --team 을 바꿔 셀을 다시 생성하세요."
    )
_SP_AUTO_DOWNLOAD = True
_SP_QUESTIONS_JSON = (
    "[[\"F01\", \"할인이나 혜택을 받기 위해 가입하고 결제한 뒤 곧바로 그만두는 일을 짧은 기간에 되풀이하면 부당 이용으로 보게 됩니다. 어느 기간 안에 몇 번 이상 반복한 경우인가요?\"], [\"F02\", \"성인용으로 분류된 콘텐츠를 보려 합니다. 이용 가능한 나이, 그 나이에 아직 못 미쳐도 예외가 되는 사람, 그리고 거쳐야 하는 본인 확인 절차를 함께 알려 주세요.\"], [\"F03\", \"한 달 넘게 제공되는 유료 상품을 중간에 그만두면 환불액에서 어떤 금액들이 빠지나요? 서비스를 시작한 뒤 일주일 안에 취소할 때는 공제 방식이 어떻게 달라지나요?\"], [\"F04\", \"한 번 사면 사용기한이 끝나지 않는 꾸미기 아이템을 환불하려고 합니다. 구매 후 언제까지 가능한지와 사용한 기간을 반영한 환불액 계산식을 약관에 적힌 표기 그대로 알려 주세요.\"], [\"F05\", \"회사가 돈을 돌려줘야 할 상황이 생긴 뒤 처리해야 하는 기한은 얼마인가요? 그보다 늦어지면 적용되는 연간 이자율도 알려 주세요.\"], [\"F06\", \"전자서명에 쓰는 인증서를 새로 발급받으려 합니다. 한 사람의 계정·기기별 발급 한도는 어떻게 되고, 다른 휴대전화나 다른 계정에서 다시 받으면 전에 쓰던 인증서는 어떻게 되나요?\"], [\"F07\", \"음성 대화 중 신고된 사람의 목소리 기록은 어느 시점부터 얼마나 보관한 뒤 없애나요?\"], [\"F08\", \"회사가 내 위치를 법에 어긋나게 다뤄 손해를 입었습니다. 어떤 법 조항 위반을 근거로 배상을 요구할 수 있고, 회사가 책임을 피하려면 무엇을 증명해야 하나요?\"], [\"F09\", \"내 위치 사용과 관련한 다툼을 회사와 합의하지 못했습니다. 방송통신위원회 절차와 개인정보 분쟁조정 절차를 각각 이용할 때 근거가 되는 법 조항은 무엇인가요?\"], [\"F10\", \"내 위치를 활용해 검색 결과나 생활 정보를 보여 주고, 사진 등에 장소를 붙이며, 다른 사람과 위치를 나누거나 광고를 제공하는 기능은 약관에서 모두 몇 가지 서비스로 구분되나요? 각 서비스의 명칭도 알려 주세요.\"], [\"F11\", \"새 Daum 아이디는 보통 어떤 과정을 거쳐 만들어지나요? 별도로 Daum 회원가입을 하지 않고 카카오계정으로 처음 이용할 때는 아이디 생성 방식이 어떻게 달라질 수 있나요?\"], [\"F12\", \"앱에서 쓰는 디지털카드는 본인이 신청하지 않아도 생길 수 있나요? 가능한 발급 경로와 발급 때 추가로 요구될 수 있는 절차, 해외 거주자나 외국인에게 생길 수 있는 제한을 알려 주세요.\"], [\"F13\", \"앱에서 발급받은 디지털카드는 카카오계정을 없앴을 때 어떻게 되나요? 발급기관 요청으로 회수된 경우 카드와 그 안의 정보를 다시 복구할 수 있는지도 알려 주세요.\"], [\"F14\", \"오랫동안 로그인하거나 접속하지 않은 계정은 회사가 어떤 연락 수단으로 먼저 알리고, 이후 계정 정보를 어떻게 처리할 수 있나요? 이 처리 기준을 자세히 보려면 어떤 운영정책을 확인해야 하나요?\"], [\"F15\", \"수사기관이 내 서비스가 금융사기에 악용되고 있다며 긴급히 막아 달라고 요청했습니다. 회사가 할 수 있는 제한, 이용자가 이의를 제기할 곳, 제한이 풀리는 조건을 알려 주세요.\"], [\"F16\", \"규칙 위반이 쌓이면 이용 제한 수위는 보통 어떤 순서로 높아지나요? 반복 횟수를 따지지 않고 곧바로 영구 제한할 수 있는 행위도 알려 주세요.\"], [\"F17\", \"카카오계정과 통합 이용계약은 그대로 두고 특정 서비스만 그만두면 이용기록과 내가 쓴 글은 언제 어떻게 되나요? 다른 사람이 공유한 글이나 남의 글에 단 댓글은 어떻게 해야 하며, 직접 삭제할 수 없는 경우도 있나요?\"], [\"F18\", \"전자서명에 쓰는 비밀정보나 인증서가 없어졌거나 새어 나갔을 가능성을 알게 됐다면 즉시 무엇을 해야 하나요? 그 정보와 인증서를 다른 사람에게 넘기거나 쓰게 하는 행위 중 금지되는 것도 알려 주세요.\"], [\"F19\", \"내 위치를 쓰거나 다른 곳에 제공하는 일을 잠시 멈춰 달라고 요청했습니다. 회사가 이 요청을 거절할 수 있는지와 요청을 처리하기 위해 갖춰야 할 수단을 알려 주세요.\"], [\"F20\", \"내 위치가 내가 지정한 다른 사람이나 업체에 전달될 때, 회사는 누구에게 어디로 어떤 내용을 언제 알려야 하나요? 위치를 모은 기기에서 문자·통화·영상을 받을 수 없다면 알림 방법은 어떻게 달라지나요?\"], [\"F21\", \"전자서명용 인증서의 발급을 제한하거나, 신청한 사람의 동의 없이 이미 발급된 인증서를 없앨 수 있는 다섯 가지 사유를 알려 주세요.\"], [\"F22\", \"카카오는 위치사업자에게 받은 위치를 이용해 현재 장소를 다른 사람과 나누고, 주변 생활·광고 정보를 보여 주며, 사진에 담긴 장소 정보로 콘텐츠를 공유하도록 돕습니다. 이 세 기능을 약관에서는 각각 어떤 서비스라고 부르며, 각 서비스는 무엇을 의미하나요?\"], [\"F23\", \"위치 서비스를 담당하는 회사의 대표전화와 위치정보 관리 책임자의 이름을 알려 주세요.\"], [\"F24\", \"카카오계정을 없애면 별도로 신청하지 않아도 함께 끝나는 이용계약은 무엇이며, 어떻게 처리되나요?\"], [\"F25\", \"앱 속 디지털카드를 받아 확인한 뒤 자기 업무나 영업에 쓰는 제3자를 약관에서는 무엇이라고 부르나요?\"], [\"F26\", \"카카오계정 회원자격정지 조치를 받은 사람이 조치 기간 중 카카오계정 이용계약을 스스로 해지한 뒤 다시 이용하겠다고 신청하면, 회사는 언제까지 승낙을 미루거나 거절할 수 있나요?\"], [\"F27\", \"카카오계정을 회사와 함께 제공하기로 제휴한 법인을 약관에서는 무엇이라고 부르며, 어떻게 정의하나요?\"], [\"F28\", \"서비스가 AI로 만든 결과물을 이용자에게 보여 줄 때 회사는 법에 따라 어떤 안내와 표시를 해야 하나요?\"], [\"F29\", \"서비스를 통해 이용자끼리, 또는 이용자와 제3자 사이에 다툼이 생겼지만 회사 잘못은 없습니다. 회사가 분쟁에 끼어들거나 손해를 물어줘야 하나요?\"], [\"F30\", \"제휴사가 발급한 앱 속 디지털카드의 내용이 맞고 적법하다는 점을 카카오가 확인하거나 보증하나요?\"]]"
)
_SP_QUESTIONS = [tuple(_x) for _x in _sp_json.loads(_SP_QUESTIONS_JSON)]
_SP_ALLOWED_DOCS = _sp_json.loads("[\"카카오계정 약관\", \"카카오 통합서비스약관\", \"카카오 통합 약관\", \"카카오 위치정보 이용약관\"]")
_SP_PER_Q_TIMEOUT_S = 120
_SP_TRANSPORT = "http"
_SP_HTTP_HOST = "127.0.0.1"
_SP_HTTP_PORT = 8765
_SP_HTTP_STARTUP_TIMEOUT_S = 30
_SP_HTTP_HEALTH_PATH = "/health"
_SP_HTTP_ANSWER_PATH = "/answer"
_SP_PERFORMANCE_REQUESTS = 12
_SP_PERFORMANCE_CONCURRENCY = 2
_SP_PERFORMANCE_REPETITIONS = 3
_SP_PERFORMANCE_WARMUP_REQUESTS = 2

_sp_fn = globals().get("answer_question")
if not callable(_sp_fn):
    raise RuntimeError(
        "팀 코드에 answer_question(question) 함수가 없습니다(규정 ②). 실행을 중단합니다."
    )

_sp_doc_warnings = []
_sp_timeouts = []
_sp_http_server = None
_sp_http_thread = None

class _SpHttpTimeout(Exception):
    """HTTP 요청 시간 초과. 품질 추출에서는 timeout_qids로 기록한다."""

def _sp_http_url(path):
    return "http://" + _SP_HTTP_HOST + ":" + str(_SP_HTTP_PORT) + path

def _sp_http_json(method, path, payload=None, timeout_s=None):
    data = None
    headers = {"Accept": "application/json"}
    if payload is not None:
        data = _sp_json.dumps(payload, ensure_ascii=False).encode("utf-8")
        headers["Content-Type"] = "application/json"
    req = _sp_urlrequest.Request(
        _sp_http_url(path), data=data, headers=headers, method=method
    )
    try:
        with _sp_urlrequest.urlopen(req, timeout=timeout_s or _SP_PER_Q_TIMEOUT_S) as resp:
            raw = resp.read().decode("utf-8")
            if resp.status != 200:
                raise RuntimeError("HTTP " + str(resp.status) + ": " + raw[:500])
    except (_sp_socket.timeout, TimeoutError) as exc:
        raise _SpHttpTimeout(str(timeout_s or _SP_PER_Q_TIMEOUT_S) + "초 안에 응답하지 않았습니다.") from exc
    except _sp_urlerror.HTTPError as exc:
        raw = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError("HTTP " + str(exc.code) + ": " + raw[:500]) from exc
    except _sp_urlerror.URLError as exc:
        if isinstance(exc.reason, (_sp_socket.timeout, TimeoutError)):
            raise _SpHttpTimeout(
                str(timeout_s or _SP_PER_Q_TIMEOUT_S) + "초 안에 응답하지 않았습니다."
            ) from exc
        raise RuntimeError("HTTP 연결 실패: " + str(exc.reason)) from exc
    try:
        return _sp_json.loads(raw)
    except _sp_json.JSONDecodeError as exc:
        raise TypeError("HTTP 응답이 JSON이 아닙니다: " + raw[:500]) from exc

def _sp_start_http_server():
    global _sp_http_server, _sp_http_thread
    _sp_app = globals().get("app")
    if _sp_app is None:
        raise RuntimeError(
            "HTTP 실행 모드에는 전역 FastAPI app과 GET /health, POST /answer가 필요합니다."
        )
    try:
        import threading as _sp_threading
        import uvicorn as _sp_uvicorn
    except ImportError as exc:
        raise RuntimeError(
            "HTTP 실행 모드에는 fastapi와 uvicorn이 필요합니다. 팀 설치 목록에 추가하세요."
        ) from exc
    _sp_config = _sp_uvicorn.Config(
        _sp_app,
        host=_SP_HTTP_HOST,
        port=_SP_HTTP_PORT,
        workers=1,
        log_level="warning",
        access_log=False,
    )
    _sp_http_server = _sp_uvicorn.Server(_sp_config)
    _sp_http_thread = _sp_threading.Thread(
        target=_sp_http_server.run, name="ktb-fastapi", daemon=True
    )
    _sp_http_thread.start()
    _sp_deadline = _sp_time.time() + _SP_HTTP_STARTUP_TIMEOUT_S
    _sp_last = None
    while _sp_time.time() < _sp_deadline:
        if not _sp_http_thread.is_alive():
            raise RuntimeError("FastAPI 서버가 준비되기 전에 종료됐습니다.")
        try:
            health = _sp_http_json("GET", _SP_HTTP_HEALTH_PATH, timeout_s=1)
            if isinstance(health, dict):
                _sp_print("[서버] FastAPI /health 준비 완료: " + _sp_http_url(_SP_HTTP_HEALTH_PATH))
                return
        except Exception as exc:
            _sp_last = exc
        _sp_time.sleep(0.2)
    _sp_stop_http_server()
    raise RuntimeError(
        "FastAPI 서버가 " + str(_SP_HTTP_STARTUP_TIMEOUT_S)
        + "초 안에 준비되지 않았습니다: " + str(_sp_last)
    )

def _sp_stop_http_server():
    if _sp_http_server is not None:
        _sp_http_server.should_exit = True
    if _sp_http_thread is not None and _sp_http_thread.is_alive():
        _sp_http_thread.join(timeout=5)

def _sp_invoke(question):
    if _SP_TRANSPORT == "http":
        return _sp_http_json(
            "POST", _SP_HTTP_ANSWER_PATH, {"question": question},
            timeout_s=_SP_PER_Q_TIMEOUT_S,
        )
    return _sp_call_with_timeout(_sp_fn, question, _SP_PER_Q_TIMEOUT_S)

if _SP_TRANSPORT == "http":
    _sp_start_http_server()

_sp_env_warnings = []
for _sp_d in ("/content/drive", "/content/gdrive", "/gdrive"):
    if _sp_os_rt.path.ismount(_sp_d):
        _sp_env_warnings.append(_sp_d + " 가 마운트되어 있습니다")
if _sp_env_warnings:
    _sp_print("", flush=True)
    _sp_print("!" * 86, flush=True)
    _sp_print("[규정 ③ 경고] 이 세션은 운영진 실행 환경과 다릅니다.", flush=True)
    for _sp_w in _sp_env_warnings:
        _sp_print("  · " + _sp_w, flush=True)
    _sp_print("  운영진은 드라이브가 연결되지 않은 새 세션에서 실행합니다. 드라이브에 둔 약관·인덱스를", flush=True)
    _sp_print("  읽고 있다면 본선에서 전량 실패합니다. 약관은 실행 중 내려받거나 셀 안에 포함하세요.", flush=True)
    _sp_print("  확인 방법: 새 노트북을 열어 코드와 이 셀만 붙여 넣고 실행해 보세요.", flush=True)
    _sp_print("!" * 86, flush=True)
    _sp_print("", flush=True)

class _SpTimeout(BaseException):
    """문항 단위 시간 초과.

    **BaseException 을 상속하는 것이 핵심이다.** 팀 코드가 `try/except Exception` 으로
    넓게 감싸는 일은 흔한데, Exception 을 상속하면 그 handler 가 시간 초과를 삼켜
    상한이 무력화된다(그대로 다음 루프를 돌며 계속 매달린다).
    """

def _sp_call_with_timeout(fn, arg, seconds):
    """SIGALRM 으로 문항 호출에 상한을 건다.

    메인 스레드가 아니거나 SIGALRM 이 없는 환경(윈도 등)에서는 signal 설정이
    실패하므로, 그때는 상한 없이 그대로 호출한다 — 상한을 못 걸었다고 해서
    채점 자체를 포기하는 편이 더 나쁘다.

    웹 Colab 셀은 IPython 이 메인 스레드에서 실행하므로 정상 동작한다.
    """
    if not seconds or seconds <= 0:
        return fn(arg)
    _sp_secs = max(1, int(seconds))     # alarm() 은 정수만 받는다. 0 은 '취소' 라 최소 1초.

    def _sp_on_alarm(signum, frame):
        raise _SpTimeout(str(_sp_secs) + "초 안에 응답하지 않았습니다.")

    try:
        _sp_prev = _sp_signal.signal(_sp_signal.SIGALRM, _sp_on_alarm)
        _sp_signal.alarm(_sp_secs)
    except (ValueError, AttributeError, OSError):
        return fn(arg)          # 상한을 걸 수 없는 환경 — 그대로 실행
    try:
        return fn(arg)
    finally:
        _sp_signal.alarm(0)
        try:
            _sp_signal.signal(_sp_signal.SIGALRM, _sp_prev)
        except Exception:
            pass

def _sp_json_safe_art(art):
    """조번호를 JSON 으로 쓸 수 있는 값으로. 표기는 최대한 원본을 살린다.

    **여기서 흡수하지 않으면 30문항을 다 돌린 뒤 파일 저장에서 터진다.**
    일부 수치 라이브러리의 정수형은 dict 도 아니고 2원소 검사도 통과하지만
    json.dump 가 거부한다. 이 값을 흡수하지 않으면
    실패 시점이 맨 끝이라 GPU 시간을 다 쓰고 결과 파일이 없는 최악의 형태가 된다.

    '제7조' 같은 문자열은 그대로 둔다 — 채점기 _art_no 가 정수로 읽는다.
    """
    if isinstance(art, bool):        # bool 은 int 의 하위형이라 먼저 걸러 낸다
        return str(art)
    if isinstance(art, (int, str)):
        return art
    try:                              # np.int64 등 정수로 볼 수 있는 것
        return int(art)
    except (TypeError, ValueError):
        return str(art)

def _sp_norm_doc(x):
    """문서명 대조용 정규화 — NFC 통일 + 공백 전부 제거.

    ⚠️ 채점기 judge_service/engine/objective.py 의 `norm_doc` 과 **같은 규칙이어야 한다.**
    러너는 Colab 셀이라 judge_service 를 import 할 수 없어 규칙을 여기에 복제해 둔다.
    한쪽만 바뀌어 어긋나면 곧바로 오탐이 난다 — 예전에 러너가 완전 일치로 대조하던 때
    '카카오계정약관'·'카카오 계정 약관' 은 실제 채점 MRR 이 1.00 인데도 규정 ④ 위반 경고를
    맞았다. 팀은 없는 문제를 고치러 다니고(자가 확인표가 n_doc_violations == 0 을 요구한다),
    정상 팀이 경고를 맞기 시작하면 아무도 경고를 안 보게 된다.
    두 구현의 일치는 submission_pipeline/tests/test_doc_name_normalization.py 가 고정한다.
    """
    return _sp_re.sub(r"\s+", "", _sp_unicodedata.normalize("NFC", str(x)))

_SP_ALLOWED_DOCS_NORM = _sp_builtins.set(_sp_norm_doc(_d) for _d in _SP_ALLOWED_DOCS)

def _sp_normalize_retrieved(qid, value):
    """retrieved 를 근거순 [[문서명, 조번호], ...] 1~4개로 정규화."""
    if not isinstance(value, (list, tuple)):
        raise TypeError(qid + ": retrieved 는 목록이어야 합니다. (실제: " + type(value).__name__ + ")")
    out = []
    for item in value:
        if isinstance(item, dict) and "doc" in item and "article_no" in item:
            doc, art = item["doc"], item["article_no"]
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            doc, art = item
        else:
            raise TypeError(qid + ": retrieved 항목은 [문서명, 조번호] 2원소여야 합니다. (실제: " + repr(item) + ")")
        doc = str(doc)
        if _SP_ALLOWED_DOCS_NORM and _sp_norm_doc(doc) not in _SP_ALLOWED_DOCS_NORM:
            _sp_doc_warnings.append({"qid": qid, "doc": doc})
        out.append([doc, _sp_json_safe_art(art)])
    if not 1 <= len(out) <= 4:
        raise ValueError(
            qid + ": retrieved 는 실제 답변 근거를 관련도 순으로 1~4개 반환해야 합니다. "
            "(실제: " + str(len(out)) + "개)"
        )
    return out

_sp_answers = []
_sp_errors = []
_sp_total = len(_SP_QUESTIONS)
_sp_print(
    "\n========== " + "공개" + " " + str(_sp_total)
    + "문항 실행 · " + _SP_TEAM + "팀 ==========",
    flush=True,
)
_sp_t0 = _sp_time.time()

for _sp_i, (_sp_qid, _sp_q) in enumerate(_SP_QUESTIONS, 1):
    _sp_print("[" + str(_sp_i).zfill(2) + "/" + str(_sp_total) + "] " + _sp_qid + " 실행 중 ...", flush=True)
    _sp_started = _sp_time.time()
    try:
        _sp_out = _sp_invoke(_sp_q)
        if not isinstance(_sp_out, dict):
            raise TypeError(_sp_qid + ": answer_question() 은 딕셔너리를 반환해야 합니다. (실제: "
                            + type(_sp_out).__name__ + ")")
        _sp_retrieved = _sp_normalize_retrieved(_sp_qid, _sp_out.get("retrieved"))
        _sp_answer = _sp_out.get("answer")
        if not isinstance(_sp_answer, str):
            raise TypeError(_sp_qid + ": answer 는 문자열이어야 합니다. (실제: "
                            + type(_sp_answer).__name__ + ")")
        _sp_answers.append({"qid": _sp_qid, "retrieved": _sp_retrieved, "answer": _sp_answer})
    except (_SpTimeout, _SpHttpTimeout) as _sp_exc:  # 한 문항이 세션 전체를 잡아먹지 않도록 끊는다.
        _sp_msg = "Timeout: " + str(_sp_exc)
        _sp_timeouts.append(_sp_qid)
        _sp_errors.append({"qid": _sp_qid, "error": _sp_msg})
        _sp_answers.append({"qid": _sp_qid, "retrieved": [], "answer": "", "error": _sp_msg})
        _sp_print("[시간초과] " + _sp_qid + " — " + _sp_msg, flush=True)
    except Exception as _sp_exc:  # 한 문항 실패로 30문항 전체를 잃지 않는다.
        _sp_msg = type(_sp_exc).__name__ + ": " + str(_sp_exc)
        _sp_errors.append({"qid": _sp_qid, "error": _sp_msg})
        _sp_answers.append({"qid": _sp_qid, "retrieved": [], "answer": "", "error": _sp_msg})
        _sp_print("[오류] " + _sp_qid + " — " + _sp_msg, flush=True)
        _sp_traceback.print_exc()
    finally:
        _sp_print("      (" + str(round(_sp_time.time() - _sp_started, 1)) + "s)", flush=True)

_sp_performance = None
if _SP_TRANSPORT == "http" and _SP_PERFORMANCE_REQUESTS > 0:
    from concurrent.futures import ThreadPoolExecutor as _SpThreadPoolExecutor

    def _sp_perf_one(index):
        _qid, _question = _SP_QUESTIONS[index % len(_SP_QUESTIONS)]
        started = _sp_time.perf_counter()
        try:
            value = _sp_http_json(
                "POST", _SP_HTTP_ANSWER_PATH, {"question": _question},
                timeout_s=_SP_PER_Q_TIMEOUT_S,
            )
            ok = (
                isinstance(value, dict)
                and isinstance(value.get("answer"), str)
                and isinstance(value.get("retrieved"), (list, tuple))
            )
            return {
                "ok": ok,
                "qid": _qid,
                "latency_s": round(_sp_time.perf_counter() - started, 4),
                "error": None if ok else "invalid_schema",
            }
        except Exception as exc:
            return {
                "ok": False,
                "qid": _qid,
                "latency_s": round(_sp_time.perf_counter() - started, 4),
                "error": type(exc).__name__ + ": " + str(exc),
            }

    def _sp_percentile(values, ratio):
        if not values:
            return None
        pos = min(len(values) - 1, max(0, int((len(values) - 1) * ratio)))
        return round(values[pos], 4)

    def _sp_median(values):
        values = sorted(values)
        if not values:
            return None
        middle = len(values) // 2
        if len(values) % 2:
            return values[middle]
        return (values[middle - 1] + values[middle]) / 2

    def _sp_perf_round(n_requests, repetition):
        started = _sp_time.perf_counter()
        with _SpThreadPoolExecutor(max_workers=max(1, _SP_PERFORMANCE_CONCURRENCY)) as pool:
            rows = list(pool.map(_sp_perf_one, range(n_requests)))
        wall_s = _sp_time.perf_counter() - started
        ok_rows = [row for row in rows if row["ok"]]
        latencies = sorted(row["latency_s"] for row in ok_rows)
        return {
            "repetition": repetition,
            "transport": "http",
            "requests": n_requests,
            "concurrency": _SP_PERFORMANCE_CONCURRENCY,
            "success": len(ok_rows),
            "fail": len(rows) - len(ok_rows),
            "success_rate": round(len(ok_rows) / len(rows), 4),
            "throughput_rps": round(len(ok_rows) / wall_s, 4) if wall_s else 0.0,
            "wall_s": round(wall_s, 4),
            "p50_latency_s": _sp_percentile(latencies, 0.50),
            "p95_latency_s": _sp_percentile(latencies, 0.95),
            "errors": [row for row in rows if not row["ok"]],
        }

    _sp_warmup = None
    if _SP_PERFORMANCE_WARMUP_REQUESTS > 0:
        _sp_print(
            "[성능] 워밍업 " + str(_SP_PERFORMANCE_WARMUP_REQUESTS) + "요청 실행 중 ...",
            flush=True,
        )
        _sp_warmup = _sp_perf_round(_SP_PERFORMANCE_WARMUP_REQUESTS, 0)

    _sp_perf_samples = []
    for _sp_repetition in range(1, _SP_PERFORMANCE_REPETITIONS + 1):
        _sp_print(
            "[성능] 측정 " + str(_sp_repetition) + "/"
            + str(_SP_PERFORMANCE_REPETITIONS) + " 실행 중 ...",
            flush=True,
        )
        _sp_perf_samples.append(
            _sp_perf_round(_SP_PERFORMANCE_REQUESTS, _sp_repetition)
        )

    _sp_success_median = _sp_median([row["success"] for row in _sp_perf_samples])
    _sp_fail_median = _sp_median([row["fail"] for row in _sp_perf_samples])
    _sp_p50_values = [
        row["p50_latency_s"] for row in _sp_perf_samples
        if row["p50_latency_s"] is not None
    ]
    _sp_p95_values = [
        row["p95_latency_s"] for row in _sp_perf_samples
        if row["p95_latency_s"] is not None
    ]
    _sp_performance = {
        "version": 2,
        "transport": "http",
        "requests": _SP_PERFORMANCE_REQUESTS,
        "concurrency": _SP_PERFORMANCE_CONCURRENCY,
        "success": int(_sp_success_median),
        "fail": int(_sp_fail_median),
        "success_rate": round(_sp_median(
            [row["success_rate"] for row in _sp_perf_samples]
        ), 4),
        "throughput_rps": round(_sp_median(
            [row["throughput_rps"] for row in _sp_perf_samples]
        ), 4),
        "wall_s": round(_sp_median(
            [row["wall_s"] for row in _sp_perf_samples]
        ), 4),
        "p50_latency_s": (
            round(_sp_median(_sp_p50_values), 4) if _sp_p50_values else None
        ),
        "p95_latency_s": (
            round(_sp_median(_sp_p95_values), 4) if _sp_p95_values else None
        ),
        "errors": [
            dict(error, repetition=sample["repetition"])
            for sample in _sp_perf_samples
            for error in sample["errors"]
        ],
        "summary_method": "median",
        "protocol": {
            "requests_per_run": _SP_PERFORMANCE_REQUESTS,
            "concurrency": _SP_PERFORMANCE_CONCURRENCY,
            "warmup_requests": _SP_PERFORMANCE_WARMUP_REQUESTS,
            "repetitions": _SP_PERFORMANCE_REPETITIONS,
        },
        "samples": _sp_perf_samples,
    }
    if _sp_warmup is not None:
        _sp_performance["warmup"] = _sp_warmup
    _sp_print(
        "[성능] closed-loop 중앙값 · "
        + str(_SP_PERFORMANCE_REQUESTS) + "요청 × "
        + str(_SP_PERFORMANCE_REPETITIONS) + "회 · 동시성 "
        + str(_SP_PERFORMANCE_CONCURRENCY) + " · 대표 성공 "
        + str(_sp_performance["success"]) + " · "
        + str(_sp_performance["throughput_rps"]) + " req/s · p95 "
        + str(_sp_performance["p95_latency_s"]) + "s",
        flush=True,
    )

_sp_stop_http_server()

_sp_submission = {"team": _SP_TEAM, "answers": _sp_answers}
if _sp_doc_warnings or _sp_timeouts or _sp_env_warnings or _sp_performance:
    _sp_submission["meta"] = {"doc_name_violations": _sp_doc_warnings,
                              "timeout_qids": _sp_timeouts,
                              "env_warnings": _sp_env_warnings,
                              "transport": _SP_TRANSPORT}
    if _sp_performance:
        _sp_submission["meta"]["performance"] = _sp_performance
_sp_text = _sp_json.dumps(_sp_submission, ensure_ascii=False, indent=2, default=str)
with _sp_open(_SP_OUTPUT_PATH, "w", encoding="utf-8") as _sp_f:
    _sp_f.write(_sp_text)

_sp_print("[완료] " + str(len(_sp_answers)) + "문항 저장: " + _SP_OUTPUT_PATH
      + "  (총 " + str(round(_sp_time.time() - _sp_t0, 1)) + "s)", flush=True)
if _sp_errors:
    _sp_print("[경고] 실패 문항 " + str(len(_sp_errors)) + "건: "
          + ", ".join(_e["qid"] for _e in _sp_errors), flush=True)
if _sp_doc_warnings:
    _sp_print("[경고] 규정 ④ 위반 — 허용 목록 밖 문서명 " + str(len(_sp_doc_warnings)) + "건: "
          + ", ".join(sorted(set(_w["doc"] for _w in _sp_doc_warnings)))
          + "  → 해당 항목은 검색 점수가 0으로 채점됩니다. 허용(띄어쓰기 차이는 무관): "
          + ", ".join(_SP_ALLOWED_DOCS), flush=True)

if _SP_AUTO_DOWNLOAD:
    try:
        from google.colab import files as _sp_files
        _sp_files.download(_SP_OUTPUT_PATH)
        _sp_print("[다운로드] 브라우저 다운로드를 시작했습니다: " + _SP_OUTPUT_PATH, flush=True)
    except Exception as _sp_dl_exc:
        _sp_print("[다운로드] 자동 다운로드 실패(" + type(_sp_dl_exc).__name__ + ": " + str(_sp_dl_exc)
                  + ") — 좌측 파일 탭에서 " + _SP_OUTPUT_PATH + " 를 직접 내려받으세요.", flush=True)

_sp_print("SUBMISSION_RUNNER_DONE " + _sp_json.dumps(
    {"team": _SP_TEAM, "output_path": _SP_OUTPUT_PATH, "n_answers": len(_sp_answers),
     "n_errors": len(_sp_errors), "failed_qids": [_e["qid"] for _e in _sp_errors],
     "n_doc_violations": len(_sp_doc_warnings), "timeout_qids": _sp_timeouts,
     "env_warnings": _sp_env_warnings, "transport": _SP_TRANSPORT,
     "performance": _sp_performance},
    ensure_ascii=False), flush=True)


[서버] FastAPI /health 준비 완료: http://127.0.0.1:8765/health

========== 공개 30문항 실행 · 13팀 ==========
[01/30] F01 실행 중 ...
[생성 실패] OutOfMemoryError: CUDA out of memory. Tried to allocate 1024.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 457.81 MiB is free. Including non-PyTorch memory, this process has 14.11 GiB memory in use. Of the allocated memory 13.82 GiB is allocated by PyTorch, and 156.90 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)
      (0.8s)
[02/30] F02 실행 중 ...
      (5.2s)
[03/30] F03 실행 중 ...
      (23.2s)
[04/30] F04 실행 중 ...
      (14.8s)
[05/30] F05 실행 중 ...
      (9.7s)
[06/30] F06 실행 중 ...
      (8.5s)
[07/30] F07 실행 중 ...
      (10.0s)
[08/30] F08 실행 중 ...
      (6.5s)
[09/30] F09 실행 중 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[다운로드] 브라우저 다운로드를 시작했습니다: /content/answers_public_13.json
SUBMISSION_RUNNER_DONE {"team": "13", "output_path": "/content/answers_public_13.json", "n_answers": 30, "n_errors": 0, "failed_qids": [], "n_doc_violations": 0, "timeout_qids": [], "env_warnings": [], "transport": "http", "performance": {"version": 2, "transport": "http", "requests": 12, "concurrency": 2, "success": 12, "fail": 0, "success_rate": 1.0, "throughput_rps": 0.1359, "wall_s": 88.2838, "p50_latency_s": 14.684, "p95_latency_s": 19.1617, "errors": [], "summary_method": "median", "protocol": {"requests_per_run": 12, "concurrency": 2, "warmup_requests": 2, "repetitions": 3}, "samples": [{"repetition": 1, "transport": "http", "requests": 12, "concurrency": 2, "success": 12, "fail": 0, "success_rate": 1.0, "throughput_rps": 0.1358, "wall_s": 88.3776, "p50_latency_s": 14.7092, "p95_latency_s": 19.2307, "errors": []}, {"repetition": 2, "transport": "http", "requests": 12, "concurrency": 2, "success": 12, "fail": 0, "success_r